# Understanding Unicode

In [ ]:
# a) What Unicode character does chr(0) return?
chr(0)

In [ ]:
# b) How does this character’s string representation (__repr__()) differ from its printed representation?
print(chr(0))

In [ ]:
# c) What happens when this character occurs in text? It may be helpful to play around with the 
# following in your Python interpreter and see if it matches your expectations:
"this is a test" + chr(0) + "string"
print("this is a test" + chr(0) + "string")

# Unicode Encodings

In [ ]:
# a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
# UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
# input strings.

print("UTF-8 is better for ASCII char training for comparing to UTF-16 and UTF-32, it prevents heading zeros, hence offers more efficient merging for BPE.")

In [ ]:
# b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string
# into a Unicode string. Why is this function incorrect? Provide an example of an input byte
# string that yields incorrect results.

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    print(bytestring)
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# function decodes bytes one by one, when there are multi-bytes char it would fail at first leading byte

a = '😀'.encode('utf-8')
decode_utf8_bytes_to_str_wrong(a)

In [ ]:
# c) Give a two-byte sequence that does not decode to any Unicode character(s).
print(b'\x53'.decode("utf-8"))
print(b'\xFE\xFE'.decode("utf-8"))

In [ ]:
# Input
corpus = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

# Strings represent tokens here for readability.
# An implementation would use bytes instead.

# Pre-tokenization
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "e", "s", "t"): 3,
    ("n", "e", "w", "e", "s", "t"): 6,
}

# Merge 1
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 8, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "e"): 3, ("e", "s"): 9,
    ("s", "t"): 9, ("n", "e"): 6,
    ("e", "w"): 6,
}

# Highest count: 9. Tie: ("s", "t") > ("e", "s").
# Merge ("s", "t") → "st".
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "e", "st"): 3,
    ("n", "e", "w", "e", "st"): 6,
}

# Merge 2
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 8, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "e"): 3, ("e", "st"): 9,
    ("n", "e"): 6, ("e", "w"): 6,
}

# Highest count: 9.
# Merge ("e", "st") → "est".
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 3
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "w"): 6, ("w", "est"): 6,
}

# Highest count: 7. Tie: ("o", "w") > ("l", "o").
# Merge ("o", "w") → "ow".
token_freq = {
    ("l", "ow"): 5,
    ("l", "ow", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 4
pair_count = {
    ("l", "ow"): 7, ("ow", "e"): 2,
    ("e", "r"): 2, ("w", "i"): 3,
    ("i", "d"): 3, ("d", "est"): 3,
    ("n", "e"): 6, ("e", "w"): 6,
    ("w", "est"): 6,
}

# Highest count: 7.
# Merge ("l", "ow") → "low".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 5
pair_count = {
    ("low", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "w"): 6, ("w", "est"): 6,
}

# Highest count: 6.
# Tie: ("w", "est") > ("n", "e") > ("e", "w").
# Merge ("w", "est") → "west".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "west"): 6,
}

# Merge 6
pair_count = {
    ("low", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "west"): 6,
}

# Highest count: 6. Tie: ("n", "e") > ("e", "west").
# Merge ("n", "e") → "ne".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("ne", "west"): 6,
}

# Ordered merges
merges = [
    ("s", "t"),
    ("e", "st"),
    ("o", "w"),
    ("l", "ow"),
    ("w", "est"),
    ("n", "e"),
]

# Final vocabulary:
# ["<|endoftext|>"] + all 256 byte tokens
#                  + ["st", "est", "ow", "low", "west", "ne"]

# "newest" tokenizes as ["ne", "west"].

# BPE Training on TinySotires
>  (a) Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary
> size of 10,000. Make sure to add the TinyStories <|endoftext|> special token to the
> vocabulary. Serialize the resulting vocabulary and merges to disk for further inspection.

> How much time and memory did training take? What is the longest token in the vocabulary?

> Does it make sense?

| Measurement | Result |
|---|---:|
| Training time, profiling off | **59.11 seconds** |
| Separate memory-monitored run | **64.31 seconds** |
| Sampled aggregate peak RSS | **6.51 GiB** |
| Longest token | **accomplishment**, **disappointment**, **responsibility** |

> (b) Profile your code. What part of the tokenizer training process takes the most time?

```bash
.venv/bin/python -m cs336_basics.profile_bpe \
  data/TinyStoriesV2-GPT4-train.txt \
  --num-workers 8 \
  --vocab-size 10000 \
  --profile \
  --output-dir /tmp/bpe-train-profile
```

| Stage | 4 workers | 8 workers |
|---|---:|---:|
| Chunk boundaries | 0\.92 s | 1\.17 s |
| Pre-tokenization | 103\.32 s | 93\.13 s |
| Initialize merge structures | 0\.44 s | 0\.49 s |
| BPE merging | 1\.30 s | 1\.46 s |
| **Total wall time** | **106\.01 s** | **96\.27 s** |

pretokenization takes the most time

# BPE Training on OpenWebText
>  (a) Train a byte-level BPE tokenizer on the OpenWebText dataset, using a maximum vocabulary
> size of 32,000. Serialize the resulting vocabulary and merges to disk for further inspection.
> What is the longest token in the vocabulary? Does it make sense?

```bash
ez@ez-Bellator-N176B:~/Transformer/assignment1-basics$ .venv/bin/python -m cs336_basics.profile_bpe \
  data/owt_train.txt \
  --num-workers 8 --chunk-size-mib 128 \
  --vocab-size 32000 --profile \
  --output-dir /tmp/bpe-owt-profile
{
  "input_path": "/home/ez/Transformer/assignment1-basics/data/owt_train.txt",
  "input_bytes": 11920511059,
  "requested_workers": 8,
  "chunk_size_bytes": 134217728,
  "vocabulary_size": 32000,
  "merge_count": 31743,
  "result_sha256": "a1bff1312f2441a3d5167d7dea33523a59018cb8fcd20bf417b577fed5ff187a",
  "profiling_enabled": true,
  "profile_directory": "/tmp/bpe-owt-profile/profiles-4lqprkpg",
  "wall_seconds": 625.2895096009997,
  "stage_seconds": {
    "boundaries": 5.144742760000554,
    "pretokenization": 316.88012281099964,
    "initialization": 65.42445965800016,
    "merging": 234.43855443899974,
    "total": 621.8878796680001,
    "chunks": 89.0
  },
  "parent_peak_rss_bytes": 13150056448,
  "largest_child_peak_rss_bytes": 11836444672,
  "memory_note": "OS process high-water marks, including interpreter overhead; child peak is not aggregate memory.",
  "artifact_paths": {
    "vocab": "/tmp/bpe-owt-profile/vocab.json",
    "merges": "/tmp/bpe-owt-profile/merges.json",
    "longest_tokens": "/tmp/bpe-owt-profile/longest_tokens.txt"
  },
  "longest_token_byte_length": 64,
  "longest_token_ids": [
    25822,
    25836
  ]
}
```
| Stage | Time | Share of measured stages |
|---|---:|---:|
| Find chunk boundaries | 5\.14 s | 0\.8% |
| Pre-tokenization: 89 chunks, 8 workers | 316\.88 s | 51\.0% |
| Initialize merge structures | 65\.42 s | 10\.5% |
| BPE merging | 234\.44 s | 37\.7% |

- **The longest tokens are both 64 bytes**: 64 consecutive hyphens, and `'ÃÂ'` repeated 16 times. The latter resembles corrupted text encoding. Both plausibly reflect repeated formatting or encoding artifacts in web text.


> (b) Compare and contrast the tokenizer that you get training on TinyStories versus
> OpenWebText.

- **Merging now matters substantially**, unlike TinyStories. It performed about **96\.7 million edge removals** and **58\.9 million edge additions**; these functions account for roughly **144 seconds cumulatively**, including their callees.
- **Smaller chunks allowed this run to finish.** Memory remains significant: parent peak RSS was **12\.25 GiB** and largest-child peak RSS **11\.03 GiB**. These are separate process high-water marks, not simultaneous aggregate usage.